# 🔬 Precision Fluid-Bed Pellet Preprocessing & Agglomeration Analysis
### *Depth-of-Field Blur Rejection, Overlap vs. Connection Discrimination & ML/DL Checkpoint*

---

### 📋 Executive Summary & Problem Formulation
In industrial fluid-bed coating and agglomeration systems monitored by high-magnification macro lenses:
1. **Narrow Depth of Field (DoF) & Focal Slicing**:
   - The optical setup focuses sharply on a specific depth plane / measurement volume within the fluidized bed.
   - Particles passing **in front of** or **behind** this focal volume appear as **diffuse, blurred bokeh blobs**.
   - As observed physically: blurred pellets have weak, gradual edge transitions and an artificially bloated apparent diameter (**"takes too much pixels to construct single pellet"**).
   - If not rejected during preprocessing, these out-of-focus blobs severely corrupt particle size distributions ($D_{10}, D_{50}, D_{90}$) and cause massive false-positive agglomeration counts.

2. **Core Objectives of This Preprocessing Checkpoint**:
   - 🎯 **Spatial Noise & Artifact Removal**: Mask static chamber walls, bottom container flanges, and intense illumination glare.
   - 🔍 **Depth-of-Field Blur Filter**: Measure perimeter boundary gradient, acutance, and high-frequency curvature to **strictly reject out-of-focus bokeh pellets**.
   - 🧩 **Interaction & Agglomeration Classification**:
     - 🟢 **Single Pellet (In-Focus)**: Isolated, circular, unperturbed pellet.
     - 🟠 **Connected / Agglomerated (In-Focus)**: Physically bound / fused pellets sharing a sintered solid/liquid bridge.
     - 🔵 **Overlapped (In-Focus)**: 2D line-of-sight occlusion displaying a sharp internal dividing boundary.
     - 🔴 **Blurred Bokeh (Rejected)**: Out-of-focus particles outside the measurement slice.
   - 💾 **ML / Deep Learning Ready**: Export cropped image tensors and structured metadata (`df_pellets.csv`) to feed downstream classification networks (YOLO, ResNet, SVM).

---

### 🛠️ Pipeline Architecture
```
[Raw High-Res BMP Frame (2048x1536)]
                  │
                  ▼
[Spatial ROI Selection: Stream Isolation & Wall Glare Rejection]
                  │
                  ▼
[Edge-Preserving Denoising & CLAHE / Top-Hat Contrast Normalization]
                  │
                  ▼
[Multi-Scale Watershed Segmentation & Distance Transform Peak Extraction]
                  │
                  ▼
[Depth-of-Field Sharpness & Acutance Metric Engine]
   ├─► Reject: Blurred Bokeh Pellets (Outside Focal Plane)
   └─► Retain: Crisp In-Focus Candidates
                  │
                  ▼
[Morphological & Internal Occlusion Edge Analysis]
   ├─► Single In-Focus Pellets
   ├─► Connected / Agglomerated Pellets (Physical Bridges)
   └─► Overlapped Pellets (Optical 2D Crossing)
                  │
                  ▼
[Granulometry (D10, D50, D90), Agglomeration Index & Checkpoint Export]
```

---
## 📦 Step 1: Environment Setup, Optical Calibration & Target Sample
Initialize computer vision libraries, scientific computing tools, and camera calibration parameters.

In [ ]:
import os
import sys
import cv2
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from PIL import Image
from IPython.display import display

# Configure console encoding safety
if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8')
    except Exception:
        pass

# Matplotlib visualization styling
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.labelsize'] = 10

# Optical & Sensor Calibration (Industrial Fluid-Bed Macro Lens)
FOV_WIDTH_MM   = 30.0                         # Horizontal Field of View (mm)
SENSOR_WIDTH   = 2048                         # Camera resolution width (px)
MM_PER_PX      = FOV_WIDTH_MM / SENSOR_WIDTH  # 0.01465 mm/px
UM_PER_PX      = MM_PER_PX * 1000.0           # 14.65 µm/px

# Target Sample Path (Early Agglomeration Fluid-Bed Experiment)
SAMPLE_PATH = r"C:\Users\user\Desktop\Computer Vision\Real-Data\exp 1000 back+ext light early agg\Pic_20260914110617199-244.bmp"

print("=" * 65)
print("[INIT] INDUSTRIAL FLUID-BED IMAGE PROCESSING ENGINE INITIALIZED")
print("=" * 65)
print(f" * Calibration Scale : {UM_PER_PX:.2f} um/pixel ({MM_PER_PX:.5f} mm/pixel)")
print(f" * Field of View     : {FOV_WIDTH_MM:.1f} mm across {SENSOR_WIDTH} pixels")
print(f" * Target Sample     : {os.path.basename(SAMPLE_PATH)}")
print(f" * Sample Exists     : {os.path.exists(SAMPLE_PATH)}")
print("=" * 65)

---
## 🖼️ Step 2: Sensor Loading & Spatial Region of Interest (ROI)
The raw sensor captures the entire chamber volume:
- **Left ($x < 180$)**: Direct backlight / external illumination glare.
- **Right ($x > 1380$)**: Stainless steel chamber wall with strong vertical reflections.
- **Bottom ($y > 830$)**: Horizontal mounting container shelf and downward reflection streaks.

We isolate the **Active Fluid-Bed Flow Stream** to focus all image processing computations purely on moving pellets.

In [ ]:
# Load original 8-bit grayscale BMP
raw_bgr = cv2.imread(SAMPLE_PATH)
if raw_bgr is None:
    raise FileNotFoundError(f"Could not load target image from: {SAMPLE_PATH}")

raw_gray = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2GRAY)
img_h, img_w = raw_gray.shape

# Active Fluid-Bed Stream Coordinates
ROI_X1, ROI_X2 = 180, 1380
ROI_Y1, ROI_Y2 = 40, 830

roi_gray = raw_gray[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]
roi_bgr  = raw_bgr[ROI_Y1:ROI_Y2, ROI_X1:ROI_X2]
h_roi, w_roi = roi_gray.shape

# Visualize Sensor View with Stream ROI Box
vis_sensor = cv2.cvtColor(raw_gray, cv2.COLOR_GRAY2RGB)
cv2.rectangle(vis_sensor, (ROI_X1, ROI_Y1), (ROI_X2, ROI_Y2), (0, 255, 255), 3)

# Mark rejected zones
cv2.putText(vis_sensor, "Active Fluid-Bed Stream", (ROI_X1 + 20, ROI_Y1 + 50),
            cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 255, 255), 2, cv2.LINE_AA)
cv2.putText(vis_sensor, "<- Left Glare", (20, 400),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 60, 60), 2, cv2.LINE_AA)
cv2.putText(vis_sensor, "Chamber Wall ->", (1420, 400),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 60, 60), 2, cv2.LINE_AA)
cv2.putText(vis_sensor, "Container Shelf / Flange Below", (400, 1000),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 60, 60), 2, cv2.LINE_AA)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(vis_sensor)
ax.set_title(f"Full Sensor Frame ({img_w} x {img_h}) with Active Stream ROI ({w_roi} x {h_roi})", fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Active Stream ROI isolated: {w_roi} x {h_roi} px ({w_roi*MM_PER_PX:.1f} x {h_roi*MM_PER_PX:.1f} mm)")

---
## ⚡ Step 3: Noise Filtering, Illumination Correction & Multi-Operator Edge Fields
To prepare the image for high-precision particle extraction and blur rejection:
1. **Edge-Preserving Bilateral Smoothing**: Eliminates CMOS high-frequency sensor noise while preserving exact sub-pixel particle boundaries.
2. **CLAHE (Contrast Limited Adaptive Histogram Equalization)**: Equalizes localized lighting variations across the chamber stream.
3. **White Top-Hat Transform**: Uses an elliptical structuring element matching typical pellet diameter to isolate bright pellet foregrounds from dark backgrounds.
4. **Differential Operators (Sobel, Laplacian, Canny)**:
   - **Sobel Gradient Magnitude** $\|
abla I\| = \sqrt{G_x^2 + G_y^2}$: Measures boundary edge slope (steep for sharp pellets, flat for blurred bokeh).
   - **Laplacian Curvature** $
abla^2 I$: Quantifies local second derivative texture.
   - **Canny Detector**: Pinpoints structural boundary lines and internal occlusion cuts.

In [ ]:
# 1. Edge-Preserving Bilateral Smoothing
smooth = cv2.bilateralFilter(roi_gray, d=5, sigmaColor=35, sigmaSpace=35)

# 2. CLAHE Enhancement
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = clahe.apply(smooth)

# 3. White Top-Hat Transform (isolates bright round structures ~30-45px)
kernel_tophat = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (35, 35))
tophat = cv2.morphologyEx(enhanced, cv2.MORPH_TOPHAT, kernel_tophat)

# 4. Differential Gradient Operators on Smoothed ROI
gx = cv2.Sobel(smooth, cv2.CV_64F, 1, 0, ksize=3)
gy = cv2.Sobel(smooth, cv2.CV_64F, 0, 1, ksize=3)
grad_mag = np.hypot(gx, gy)
laplacian = cv2.Laplacian(smooth, cv2.CV_64F, ksize=3)
canny = cv2.Canny(smooth, threshold1=30, threshold2=75)

# 4-Panel Preprocessing Visualizer
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].imshow(roi_gray, cmap='gray')
axes[0, 0].set_title("1. Raw Stream ROI", fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(tophat, cmap='magma')
axes[0, 1].set_title("2. Morphological Top-Hat (Pellet Contrast)", fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(grad_mag, cmap='inferno', vmin=0, vmax=80)
axes[1, 0].set_title("3. Sobel Gradient Magnitude ||grad I|| (Edge Energy)", fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(canny, cmap='gray')
axes[1, 1].set_title("4. Canny Structural Edge Map", fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("[OK] Preprocessing and differential edge fields generated successfully.")

---
## 🎯 Step 4: Multi-Scale Pellet Segmentation & Seeded Watershed
Unlike simple Hough circle detectors (which fail on touching, dumbbell, or irregular agglomerates), we employ **Seeded Distance-Transform Watershed Segmentation**:
1. **Otsu Binarization on Top-Hat Field**: Automatically segments bright candidate pellet bodies from the dark fluid-bed backdrop.
2. **Morphological Opening & Closing**: Cleans tiny spurious speckles and seals small internal holes.
3. **Euclidean Distance Transform ($D_{L2}$)**: Computes the distance from each foreground pixel to the nearest background pixel.
4. **Local Peak Detection**: Identifies true pellet centers as seeds (`markers`), even when multiple pellets are touching in an agglomerate cluster!
5. **Topological Watershed**: Expands outward from the seeds along negative distance ridges to cleanly separate contacting pellets while preserving true boundaries.

In [ ]:
# 1. Binarize Top-Hat Contrast Field
_, binary = cv2.threshold(tophat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 2. Morphological Cleanup
k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
binary_clean = cv2.morphologyEx(binary, cv2.MORPH_OPEN, k3)
binary_clean = cv2.morphologyEx(binary_clean, cv2.MORPH_CLOSE, k5)

# 3. Euclidean Distance Transform
dist_map = cv2.distanceTransform(binary_clean, cv2.DIST_L2, 5)

# 4. Extract Peak Seeds (Local Maxima)
# Minimum distance between pellet centers ~11 px (~160 µm)
coords = peak_local_max(dist_map, min_distance=11, threshold_abs=4.0)

# Build seed markers
markers_seed = np.zeros(dist_map.shape, dtype=bool)
markers_seed[tuple(coords.T)] = True
markers_labeled, num_seeds = ndi.label(markers_seed)

# 5. Seeded Watershed Segmentation
labels = watershed(-dist_map, markers_labeled, mask=binary_clean)

# Visualize Segmentation Steps
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

axes[0].imshow(binary_clean, cmap='gray')
axes[0].set_title("A. Binarized Foreground Mask", fontweight='bold')
axes[0].axis('off')

axes[1].imshow(dist_map, cmap='viridis')
axes[1].scatter(coords[:, 1], coords[:, 0], c='red', s=6, label=f'Pellet Centers ({len(coords)})')
axes[1].set_title("B. Distance Transform & Seed Peaks", fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].axis('off')

axes[2].imshow(labels, cmap='tab20b')
axes[2].set_title(f"C. Watershed Labeled Regions ({labels.max()} candidates)", fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"[OK] Total candidate pellet seeds identified: {labels.max()}")

---
## 🔍 Step 5: Depth-of-Field Blur Measurement & Rejection Engine

### 📐 The Mathematical Physics of Focus & Bokeh
When the camera focuses on a shallow Region of Interest (depth slice $\Delta z$):
- **Out-of-Focus Bokeh Pellets**: Pellets outside the focal slice suffer from the *Circle of Confusion*. Their boundary transitions are wide and diffuse, and their apparent size bloats (**"takes too much pixels to construct single pellet"**).
- **In-Focus Pellets**: Have crisp perimeter edges with a steep intensity rise, high boundary gradient, and sharp curvature.

For each segmented candidate region $P_i$:
1. **Perimeter Boundary Gradient ($S_{\text{edge}}$)**:
   Mean and 80th-percentile Sobel gradient magnitude along the outer contour ring $\partial P_i$:
   $$S_{\text{edge}} = \frac{1}{|\partial P_i|} \sum_{(x,y) \in \partial P_i} \|\nabla I(x,y)\|$$
2. **Sharp Perimeter Fraction ($R_{\text{sharp}}$)**:
   Fraction of boundary pixels exceeding the steep edge threshold ($\tau = 26.0$):
   $$R_{\text{sharp}} = \frac{1}{|\partial P_i|} \sum_{(x,y) \in \partial P_i} \mathbb{I}\big(\|\nabla I(x,y)\| > 26.0\big)$$
3. **Internal Laplacian Curvature ($\sigma_{\text{lap}}^2$)**:
   High-frequency variance inside the particle core.
4. **Composite Focus Score ($F$)**:
   $$F = 0.60 \cdot S_{\text{edge}}^{p80} + 25.0 \cdot R_{\text{sharp}} + 0.15 \cdot \sqrt{\sigma_{\text{lap}}^2}$$

**Rejection Rule**:
$$\text{Status}(P_i) = \begin{cases} \mathbf{In\text{-}Focus} & \text{if } S_{\text{edge}}^{p80} \ge 27.0 \;\wedge\; R_{\text{sharp}} \ge 0.28 \;\wedge\; F \ge 26.0 \\ \mathbf{Blurred\text{ }Bokeh\text{ (Reject)}} & \text{otherwise} \end{cases}$$

In [ ]:
# Evaluate focus & sharpness metrics for every candidate region
FOCUS_EDGE_THRESH = 27.0   # 80th-percentile gradient threshold
FOCUS_RATIO_THRESH = 0.28  # Minimum fraction of sharp boundary
COMPOSITE_THRESH   = 26.0  # Overall composite focus threshold

candidates = []

for lid in range(1, labels.max() + 1):
    mask = (labels == lid).astype(np.uint8)
    area = int(np.sum(mask))
    
    # Filter noise speckles (< 65 px) and huge background zones (> 15,000 px)
    if area < 65 or area > 15000:
        continue
        
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        continue
    cnt = cnts[0]
    
    # 1. Boundary Perimeter Gradient
    boundary_mask = cv2.morphologyEx(mask, cv2.MORPH_GRADIENT, k3)
    b_pts = boundary_mask > 0
    if not np.any(b_pts):
        continue
        
    edge_vals = grad_mag[b_pts]
    mean_edge_grad = float(np.mean(edge_vals))
    p80_edge_grad  = float(np.percentile(edge_vals, 80))
    sharp_ratio    = float(np.mean(edge_vals > 26.0))
    
    # 2. Internal Laplacian Curvature Variance
    inner_mask = cv2.erode(mask, k3)
    lap_var = float(np.var(laplacian[inner_mask > 0])) if np.any(inner_mask > 0) else 0.0
    
    # 3. Geometric Center & Moments
    M = cv2.moments(cnt)
    if M['m00'] == 0: continue
    cx = int(M['m10'] / M['m00'])
    cy = int(M['m01'] / M['m00'])
    
    # 4. Shape & Acutance Metrics
    perimeter = cv2.arcLength(cnt, True)
    circularity = float(4.0 * np.pi * area / (perimeter**2)) if perimeter > 0 else 0.0
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    solidity = float(area / hull_area) if hull_area > 0 else 1.0
    
    if len(cnt) >= 5:
        (rx, ry), (rw, rh), angle = cv2.minAreaRect(cnt)
        aspect_ratio = float(max(rw, rh) / max(1.0, min(rw, rh)))
    else:
        aspect_ratio = 1.0
        
    equiv_diam_px = float(2.0 * np.sqrt(area / np.pi))
    equiv_diam_um = equiv_diam_px * UM_PER_PX
    
    # 5. Core Peak Count inside Region
    dist_in_mask = dist_map * mask
    peaks_in_mask = peak_local_max(dist_in_mask, min_distance=8, threshold_abs=4.0)
    num_cores = len(peaks_in_mask)
    
    # 6. Composite Focus Score
    focus_score = float(0.60 * p80_edge_grad + 25.0 * sharp_ratio + 0.15 * np.sqrt(max(0, lap_var)))
    
    # Decision: In-Focus vs Blurred Bokeh
    is_in_focus = (p80_edge_grad >= FOCUS_EDGE_THRESH and 
                   sharp_ratio >= FOCUS_RATIO_THRESH and 
                   focus_score >= COMPOSITE_THRESH)
                   
    candidates.append({
        'id': lid,
        'cx': cx, 'cy': cy,
        'area_px': area,
        'equiv_diam_px': round(equiv_diam_px, 1),
        'equiv_diam_um': round(equiv_diam_um, 1),
        'mean_edge_grad': round(mean_edge_grad, 1),
        'p80_edge_grad': round(p80_edge_grad, 1),
        'sharp_ratio': round(sharp_ratio, 2),
        'lap_var': round(lap_var, 1),
        'focus_score': round(focus_score, 1),
        'aspect_ratio': round(aspect_ratio, 2),
        'solidity': round(solidity, 2),
        'circularity': round(circularity, 2),
        'num_cores': num_cores,
        'is_in_focus': is_in_focus,
        'cnt': cnt,
        'mask': mask
    })

df_all = pd.DataFrame(candidates)
df_sharp   = df_all[df_all['is_in_focus']].copy()
df_blurred = df_all[~df_all['is_in_focus']].copy()

print("=" * 65)
print("DEPTH-OF-FIELD FOCUS FILTER RESULTS:")
print(f" * Total Candidate Pellets Segmented : {len(df_all):4d}")
print(f" * In-Focus Pellets Retained         : {len(df_sharp):4d}  ({len(df_sharp)/len(df_all)*100:5.1f}%)")
print(f" * Blurred Bokeh Blobs Rejected      : {len(df_blurred):4d}  ({len(df_blurred)/len(df_all)*100:5.1f}%)")
print("=" * 65)

# Focus Score Distribution Histogram
plt.figure(figsize=(10, 4.5))
plt.hist(df_all['focus_score'], bins=35, color='#4575b4', edgecolor='black', alpha=0.8, label='Candidate Particles')
plt.axvline(COMPOSITE_THRESH, color='red', linestyle='--', linewidth=2.5, label=f'Focus Threshold ({COMPOSITE_THRESH})')
plt.fill_betweenx([0, 25], 0, COMPOSITE_THRESH, color='red', alpha=0.12, label='Blurred Bokeh (Rejected)')
plt.fill_betweenx([0, 25], COMPOSITE_THRESH, df_all['focus_score'].max() + 5, color='green', alpha=0.12, label='In-Focus Slice (Kept)')
plt.title("Distribution of Focus Scores (Depth-of-Field Blur Rejection)", fontweight='bold')
plt.xlabel("Composite Focus Score [0.60*Grad_p80 + 25.0*SharpRatio + 0.15*sqrt(LapVar)]")
plt.ylabel("Particle Count")
plt.ylim(0, 22)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🔬 Step 6: In-Focus Pellet Classification (Single vs. Connected vs. Overlapped)

### Physical Discrimination Criteria for In-Focus Pellets:
1. 🟢 **Single Pellets**:
   - Single distance transform core (`num_cores == 1`).
   - High circularity ($C \ge 0.75$), high solidity ($S \ge 0.88$), and low aspect ratio ($AR < 1.28$).
   - Freely fluidized, independent individual pellets.

2. 🟠 **Connected / Agglomerated Pellets (Physical Agglomerates)**:
   - Multiple distance transform cores (`num_cores > 1`) OR elongated dumbbell contour ($AR \ge 1.28$ or $S < 0.88$).
   - **Continuous surface bridge**: Liquid/binder or sintered solid bridge connecting the particles without an internal occluding boundary line.

3. 🔵 **Overlapped Pellets (Optical Occlusion)**:
   - Pellets passing each other at slightly different depths along the camera's optical line of sight.
   - Characterized by an **internal Canny edge** cutting through the intersection region where the foreground pellet's boundary occludes the background pellet.

In [ ]:
categories = []

for idx, row in df_sharp.iterrows():
    ar = row['aspect_ratio']
    sol = row['solidity']
    cores = row['num_cores']
    mask = row['mask']
    
    # Internal Canny Edge Analysis:
    # Erode the mask to ignore the outer contour, then check for internal occlusion edges
    inner_mask = cv2.erode(mask, k3)
    internal_canny_count = np.sum((canny > 0) & (inner_mask > 0))
    internal_canny_density = internal_canny_count / max(1, row['area_px'])
    
    if ar < 1.28 and sol >= 0.88 and cores <= 1:
        cat = 'single'
    else:
        # If an internal occlusion edge cuts across the shared area -> Overlapped
        # If continuous intensity bridge with no internal cut -> Connected / Agglomerated
        if internal_canny_density > 0.08:
            cat = 'overlapped'
        else:
            cat = 'connected'
            
    categories.append(cat)

df_sharp['category'] = categories

cat_counts = df_sharp['category'].value_counts().to_dict()
n_single = cat_counts.get('single', 0)
n_connected = cat_counts.get('connected', 0)
n_overlapped = cat_counts.get('overlapped', 0)
n_total_sharp = len(df_sharp)
agg_index = (n_connected / max(1, n_total_sharp)) * 100.0

print("=" * 65)
print("IN-FOCUS PELLET INTERACTION BREAKDOWN:")
print(f" * Single Pellets             : {n_single:3d}  ({n_single/n_total_sharp*100:5.1f}%)")
print(f" * Connected / Agglomerated   : {n_connected:3d}  ({n_connected/n_total_sharp*100:5.1f}%)")
print(f" * Overlapped (Optical 2D)    : {n_overlapped:3d}  ({n_overlapped/n_total_sharp*100:5.1f}%)")
print("-" * 65)
print(f" * Total In-Focus Pellets     : {n_total_sharp:3d}")
print(f" * Agglomeration Index        : {agg_index:.2f}% (Connected / Total Sharp)")
print("=" * 65)

---
## 🎨 Step 7: Master Annotated Overlay Visualization
Displays all detected particles mapped back onto the active fluid-bed stream:
- 🟢 **Green Solid Contours**: In-Focus Single Pellets (`S`)
- 🟠 **Orange Thick Contours**: In-Focus Connected Agglomerates (`C`)
- 🔵 **Blue Contours**: In-Focus Overlapped Pellets (`O`)
- 🔴 **Faint Red Outlines**: Blurred Bokeh Blobs (Successfully Rejected by DoF Filter)

In [ ]:
vis_master = roi_bgr.copy()

# 1. Draw Blurred Bokeh Pellets (Rejected) as subtle red outlines
for _, row in df_blurred.iterrows():
    cv2.drawContours(vis_master, [row['cnt']], -1, (0, 0, 200), 1)

COLOR_MAP = {
    'single':     (0, 255, 0),       # Green
    'connected':  (0, 165, 255),     # Orange / Amber
    'overlapped': (255, 120, 0)      # Cyan / Blue
}

# 2. Draw In-Focus Pellets with Color-Coded Category & ID Tag
for _, row in df_sharp.iterrows():
    color = COLOR_MAP[row['category']]
    cv2.drawContours(vis_master, [row['cnt']], -1, color, 2)
    cv2.circle(vis_master, (row['cx'], row['cy']), 2, color, -1)
    
    tag = row['category'][0].upper() # 'S', 'C', 'O'
    cv2.putText(vis_master, tag, (row['cx'] - 4, row['cy'] + 4),
                cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

# Display Master Figure
fig, ax = plt.subplots(figsize=(15, 10))
ax.imshow(cv2.cvtColor(vis_master, cv2.COLOR_BGR2RGB))
ax.set_title(
    f"Master Preprocessing & Focus Filter Overlay ({os.path.basename(SAMPLE_PATH)})\n"
    f"[Green: Single ({n_single}) | Orange: Connected Agglomerate ({n_connected}) | "
    f"Blue: Overlapped ({n_overlapped}) | Red: Blurred Bokeh Rejected ({len(df_blurred)})]",
    fontsize=12, fontweight='bold', pad=12
)
ax.axis('off')
plt.tight_layout()
plt.show()

---
## 🔎 Step 8: Microscopic High-Detail Diagnostic Gallery
Zoomed-in diagnostic crops comparing representative phenotypes of:
1. **Single In-Focus Pellet**: Round, crisp boundary, high gradient ring.
2. **Connected Agglomerate**: Touching pair/triplet sharing a physical bridge.
3. **Overlapped Pellets**: 2D projection crossing with internal occlusion edge.
4. **Blurred Bokeh Blob**: Diffuse, soft out-of-focus blob successfully rejected.

In [ ]:
def extract_patch(img, cx, cy, size=65):
    half = size // 2
    y1 = max(0, cy - half)
    y2 = min(img.shape[0], cy + half)
    x1 = max(0, cx - half)
    x2 = min(img.shape[1], cx + half)
    patch = img[y1:y2, x1:x2]
    return cv2.resize(patch, (160, 160), interpolation=cv2.INTER_NEAREST)

# Select representative examples from each category
ex_single = df_sharp[df_sharp['category'] == 'single'].sort_values('focus_score', ascending=False).iloc[2]
ex_conn   = df_sharp[df_sharp['category'] == 'connected'].sort_values('area_px', ascending=False).iloc[1]
ex_ovl    = df_sharp[df_sharp['category'] == 'overlapped'].iloc[0] if n_overlapped > 0 else None
ex_blur   = df_blurred.sort_values('area_px', ascending=False).iloc[len(df_blurred)//3]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))

# 1. Single
axes[0].imshow(cv2.cvtColor(extract_patch(vis_master, ex_single['cx'], ex_single['cy']), cv2.COLOR_BGR2RGB))
axes[0].set_title(f"In-Focus Single\nFocus: {ex_single['focus_score']:.1f} | D: {ex_single['equiv_diam_um']:.0f} um",
                  fontweight='bold', color='green')
axes[0].axis('off')

# 2. Connected
axes[1].imshow(cv2.cvtColor(extract_patch(vis_master, ex_conn['cx'], ex_conn['cy']), cv2.COLOR_BGR2RGB))
axes[1].set_title(f"In-Focus Connected (Agglomerate)\nFocus: {ex_conn['focus_score']:.1f} | AR: {ex_conn['aspect_ratio']:.2f}",
                  fontweight='bold', color='darkorange')
axes[1].axis('off')

# 3. Overlapped
if ex_ovl is not None:
    axes[2].imshow(cv2.cvtColor(extract_patch(vis_master, ex_ovl['cx'], ex_ovl['cy']), cv2.COLOR_BGR2RGB))
    axes[2].set_title(f"In-Focus Overlapped (2D Occlusion)\nFocus: {ex_ovl['focus_score']:.1f}",
                      fontweight='bold', color='navy')
axes[2].axis('off')

# 4. Blurred
axes[3].imshow(cv2.cvtColor(extract_patch(vis_master, ex_blur['cx'], ex_blur['cy']), cv2.COLOR_BGR2RGB))
axes[3].set_title(f"Blurred Bokeh (Rejected)\nFocus: {ex_blur['focus_score']:.1f} < {COMPOSITE_THRESH}",
                  fontweight='bold', color='crimson')
axes[3].axis('off')

plt.suptitle("Microscopic Phenotype Verification Cards", fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

---
## 📊 Step 9: Granulometric Particle Size Distribution ($D_{10}, D_{50}, D_{90}$)
Computes the industry-standard cumulative particle size distribution (PSD) for the verified in-focus pellets:
- **$D_{10}$**: 10% of particles are smaller than this diameter.
- **$D_{50}$**: Median particle diameter.
- **$D_{90}$**: 90% of particles are smaller than this diameter.
- **Relative Span**: $(D_{90} - D_{10}) / D_{50}$ (indicator of size polydispersity).

In [ ]:
diams_um = np.sort(df_sharp['equiv_diam_um'].values)
d10 = np.percentile(diams_um, 10)
d50 = np.percentile(diams_um, 50)
d90 = np.percentile(diams_um, 90)
span = (d90 - d10) / d50

print("=" * 65)
print("INDUSTRIAL GRANULOMETRY & PARTICLE SIZE DISTRIBUTION (PSD):")
print(f" * D10 Diameter           : {d10:6.1f} um ({d10/1000.0:.3f} mm)")
print(f" * D50 (Median Diameter)  : {d50:6.1f} um ({d50/1000.0:.3f} mm)")
print(f" * D90 Diameter           : {d90:6.1f} um ({d90/1000.0:.3f} mm)")
print(f" * Polydispersity Span    : {span:6.2f}")
print("=" * 65)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# 1. Cumulative PSD Histogram
ax1.hist(diams_um, bins=18, color='#3182bd', edgecolor='black', alpha=0.75)
ax1.axvline(d10, color='orange', linestyle='--', linewidth=2, label=f'D10 = {d10:.0f} um')
ax1.axvline(d50, color='red', linestyle='-', linewidth=2.5, label=f'D50 = {d50:.0f} um (Median)')
ax1.axvline(d90, color='purple', linestyle='--', linewidth=2, label=f'D90 = {d90:.0f} um')
ax1.set_title("Particle Size Distribution (In-Focus Pellets)", fontweight='bold')
ax1.set_xlabel("Equivalent Spherical Diameter (um)")
ax1.set_ylabel("Pellet Count")
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# 2. Category Breakdown Bar Chart
bars = ax2.bar(['Single', 'Connected\n(Agglomerate)', 'Overlapped'],
               [n_single, n_connected, n_overlapped],
               color=['#2ca02c', '#ff7f0e', '#1f77b4'], edgecolor='black', alpha=0.85)

for bar in bars:
    y = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2.0, y + 1.0,
             f"{int(y)} ({y/n_total_sharp*100:.1f}%)",
             ha='center', va='bottom', fontweight='bold')

ax2.set_title("In-Focus Pellet Interaction Breakdown", fontweight='bold')
ax2.set_ylabel("Pellet Count")
ax2.set_ylim(0, max(n_single, n_connected) * 1.25)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSample of First 5 In-Focus Extracted Pellets:")
display(df_sharp[['id', 'cx', 'cy', 'equiv_diam_um', 'focus_score', 'aspect_ratio', 'solidity', 'category']].head())

---
## 💾 Step 10: Machine Learning & Deep Learning Checkpoint Exporter
Export the clean, preprocessed in-focus crops and structured ground-truth metadata to disk.
This forms the **bridge checkpoint** for downstream training:
- **`preprocessed_checkpoint/crops/single/`**: High-detail normalized image crops of single in-focus pellets.
- **`preprocessed_checkpoint/crops/connected/`**: High-detail normalized image crops of connected agglomerates.
- **`preprocessed_checkpoint/crops/overlapped/`**: High-detail normalized image crops of overlapped pairs.
- **`preprocessed_checkpoint/preprocessed_pellets.csv`**: Full geometric and optical feature table for tabular ML (RandomForest, XGBoost, SVM).

In [ ]:
CHECKPOINT_DIR = "preprocessed_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
for sub in ['single', 'connected', 'overlapped']:
    os.makedirs(os.path.join(CHECKPOINT_DIR, 'crops', sub), exist_ok=True)

# 1. Save Structured CSV Table
csv_export = df_sharp[['id', 'cx', 'cy', 'area_px', 'equiv_diam_px', 'equiv_diam_um',
                        'p80_edge_grad', 'mean_edge_grad', 'sharp_ratio', 'lap_var',
                        'focus_score', 'aspect_ratio', 'solidity', 'circularity',
                        'num_cores', 'category']].copy()

csv_path = os.path.join(CHECKPOINT_DIR, "preprocessed_pellets.csv")
csv_export.to_csv(csv_path, index=False)
print(f"[OK] Metadata saved to: {csv_path} ({len(csv_export)} pellets)")

# 2. Export Square Crops (64x64) for Deep Learning Backbones (PyTorch / ResNet / YOLO)
CROP_DIM = 64
exported_count = 0

for _, row in df_sharp.iterrows():
    cx, cy = row['cx'], row['cy']
    cat = row['category']
    r = int(row['equiv_diam_px'] / 2.0 * 1.35)  # 35% contextual margin
    
    y1 = max(0, cy - r)
    y2 = min(h_roi, cy + r)
    x1 = max(0, cx - r)
    x2 = min(w_roi, cx + r)
    
    crop = roi_bgr[y1:y2, x1:x2]
    if crop.size > 0 and crop.shape[0] >= 6 and crop.shape[1] >= 6:
        crop_resized = cv2.resize(crop, (CROP_DIM, CROP_DIM), interpolation=cv2.INTER_LINEAR)
        out_filename = f"pellet_{row['id']}_{cat}.png"
        out_file = os.path.join(CHECKPOINT_DIR, 'crops', cat, out_filename)
        cv2.imwrite(out_file, crop_resized)
        exported_count += 1

print(f"[OK] Exported {exported_count} normalized ({CROP_DIM}x{CROP_DIM}) crops for Deep Learning.")

---
## 🚀 Step 11: Reusable Processing Function for Batch Operations
A self-contained, modular function `preprocess_fluidbed_frame` to process any arbitrary BMP frame from any experimental run in `Real-Data`.

In [ ]:
def preprocess_fluidbed_frame(image_path,
                               roi=(180, 1380, 40, 830),
                               focus_edge_thresh=27.0,
                               sharp_ratio_thresh=0.28,
                               composite_thresh=26.0):
    """
    End-to-End Image Processing & Depth-of-Field Blur Rejection Function.
    
    Parameters:
    -----------
    image_path         : str, Path to raw BMP frame
    roi                : tuple, (x1, x2, y1, y2) bounds of active stream
    focus_edge_thresh  : float, 80th-percentile edge gradient threshold (default: 27.0)
    sharp_ratio_thresh : float, Minimum fraction of sharp boundary (default: 0.28)
    composite_thresh   : float, Overall focus score cutoff (default: 26.0)
    
    Returns:
    --------
    dict containing:
      - 'df_sharp'     : pd.DataFrame of in-focus pellets
      - 'df_blurred'   : pd.DataFrame of rejected bokeh blobs
      - 'counts'       : dict of category counts
      - 'agg_index'    : float, percentage of connected pellets
      - 'psd'          : dict of D10, D50, D90 in micrometers
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot read: {image_path}")
        
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    x1, x2, y1, y2 = roi
    roi_gray = gray[y1:y2, x1:x2]
    h_r, w_r = roi_gray.shape
    
    smooth = cv2.bilateralFilter(roi_gray, 5, 35, 35)
    gx = cv2.Sobel(smooth, cv2.CV_64F, 1, 0, 3)
    gy = cv2.Sobel(smooth, cv2.CV_64F, 0, 1, 3)
    grad = np.hypot(gx, gy)
    lap = cv2.Laplacian(smooth, cv2.CV_64F, 3)
    canny_map = cv2.Canny(smooth, 30, 75)
    
    clahe_obj = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enh = clahe_obj.apply(smooth)
    k_th = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (35, 35))
    th = cv2.morphologyEx(enh, cv2.MORPH_TOPHAT, k_th)
    
    _, bin_f = cv2.threshold(th, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k_e = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    k_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    bin_clean = cv2.morphologyEx(bin_f, cv2.MORPH_OPEN, k_e)
    bin_clean = cv2.morphologyEx(bin_clean, cv2.MORPH_CLOSE, k_c)
    
    dist_m = cv2.distanceTransform(bin_clean, cv2.DIST_L2, 5)
    seeds = peak_local_max(dist_m, min_distance=11, threshold_abs=4.0)
    m_seed = np.zeros(dist_m.shape, dtype=bool)
    m_seed[tuple(seeds.T)] = True
    m_lbl, _ = ndi.label(m_seed)
    lbls = watershed(-dist_m, m_lbl, mask=bin_clean)
    
    records = []
    for l in range(1, lbls.max() + 1):
        m = (lbls == l).astype(np.uint8)
        area = int(np.sum(m))
        if area < 65 or area > 15000: continue
        
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: continue
        cnt = cnts[0]
        
        bnd = cv2.morphologyEx(m, cv2.MORPH_GRADIENT, k_e)
        if not np.any(bnd > 0): continue
        
        e_vals = grad[bnd > 0]
        p80_g = float(np.percentile(e_vals, 80))
        sh_rat = float(np.mean(e_vals > 26.0))
        
        in_m = cv2.erode(m, k_e)
        l_var = float(np.var(lap[in_m > 0])) if np.any(in_m > 0) else 0.0
        
        M_c = cv2.moments(cnt)
        if M_c['m00'] == 0: continue
        cx_p = int(M_c['m10'] / M_c['m00'])
        cy_p = int(M_c['m01'] / M_c['m00'])
        
        if len(cnt) >= 5:
            _, (rw, rh), _ = cv2.minAreaRect(cnt)
            ar_p = float(max(rw, rh) / max(1.0, min(rw, rh)))
        else:
            ar_p = 1.0
            
        hl = cv2.convexHull(cnt)
        hl_a = cv2.contourArea(hl)
        sol_p = float(area / hl_a) if hl_a > 0 else 1.0
        
        dist_in = dist_m * m
        n_cores = len(peak_local_max(dist_in, min_distance=8, threshold_abs=4.0))
        
        f_score = float(0.60 * p80_g + 25.0 * sh_rat + 0.15 * np.sqrt(max(0, l_var)))
        in_foc = (p80_g >= focus_edge_thresh and sh_rat >= sharp_ratio_thresh and f_score >= composite_thresh)
        
        records.append({
            'id': l, 'cx': cx_p, 'cy': cy_p, 'area_px': area,
            'equiv_diam_um': round(2.0 * np.sqrt(area / np.pi) * UM_PER_PX, 1),
            'focus_score': round(f_score, 1), 'aspect_ratio': round(ar_p, 2),
            'solidity': round(sol_p, 2), 'num_cores': n_cores,
            'is_in_focus': in_foc, 'mask': m
        })
        
    df_res = pd.DataFrame(records)
    if len(df_res) == 0:
        return {'df_sharp': pd.DataFrame(), 'agg_index': 0.0}
        
    df_sh = df_res[df_res['is_in_focus']].copy()
    df_bl = df_res[~df_res['is_in_focus']].copy()
    
    cats = []
    for _, r in df_sh.iterrows():
        in_m = cv2.erode(r['mask'], k_e)
        canny_cnt = np.sum((canny_map > 0) & (in_m > 0))
        if r['aspect_ratio'] < 1.28 and r['solidity'] >= 0.88 and r['num_cores'] <= 1:
            cats.append('single')
        else:
            cats.append('overlapped' if (canny_cnt / max(1, r['area_px'])) > 0.08 else 'connected')
            
    df_sh['category'] = cats
    counts = df_sh['category'].value_counts().to_dict() if len(df_sh) > 0 else {}
    n_conn = counts.get('connected', 0)
    agg_idx = (n_conn / max(1, len(df_sh))) * 100.0
    
    diams = np.sort(df_sh['equiv_diam_um'].values) if len(df_sh) > 0 else np.array([0])
    psd_vals = {
        'D10': np.percentile(diams, 10),
        'D50': np.percentile(diams, 50),
        'D90': np.percentile(diams, 90)
    }
    
    return {
        'df_sharp': df_sh,
        'df_blurred': df_bl,
        'counts': counts,
        'agg_index': agg_idx,
        'psd': psd_vals
    }

# Test Batch Function on Next Frame
next_sample = r"C:\Users\user\Desktop\Computer Vision\Real-Data\exp 1000 back+ext light early agg\Pic_20260914110617209-245.bmp"
res_batch = preprocess_fluidbed_frame(next_sample)
print("Batch Run on Frame 245:")
print(f" * In-Focus Pellets   : {len(res_batch['df_sharp'])}")
print(f" * Blurred Rejected   : {len(res_batch['df_blurred'])}")
print(f" * Classification     : {res_batch['counts']}")
print(f" * Agglomeration Index: {res_batch['agg_index']:.2f}%")
print(f" * Median Size (D50)  : {res_batch['psd']['D50']:.1f} um")